In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, time, zipfile, psutil
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[key]='2'
os.environ['CUDA_VISIBLE_DEVICES']='-1'
target=pathlib.Path('/tmp/exp053-inputs')
target.mkdir(exist_ok=False)
raw=base64.b64decode('...',validate=True)
assert hashlib.sha256(raw).hexdigest()=='bc919576c977dcf6a529ad8c385c222bea2f058b4d8601d2e44ee4fd8e10c0ef'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        assert pathlib.PurePosixPath(name).name==name and ':' not in name
        (target/name).write_bytes(z.read(name))
def digest(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
cfg=json.loads((target/'experiment-config.json').read_text())
assert cfg['job']=='main' and cfg['experiment_id']=='EXP-053'
assert not pathlib.Path('/dev/nvidia0').exists()
for name in cfg['mounted']:
    matches=[p for p in pathlib.Path('/kaggle/input').rglob(name) if digest(p)==cfg['input_hashes'][name]]
    assert matches, ('Missing expected input bytes',name)
    (target/name).write_bytes(matches[0].read_bytes())
for name,d in cfg['input_hashes'].items():
    assert digest(target/name)==d,name
import numpy,pandas,scipy,sklearn
actual={'numpy':numpy.__version__,'pandas':pandas.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__}
assert actual==cfg['required_versions'], (actual,cfg['required_versions'])
out=pathlib.Path('/kaggle/working/results')
out.mkdir(exist_ok=False)
start=time.monotonic()
peak=0
child=subprocess.Popen([sys.executable,'-u',str(target/'main_cpu.py'),'--input',str(target),'--output',str(out)])
try:
    process=psutil.Process(child.pid)
    next_print=0
    while child.poll() is None:
        rss=0
        try:
            for p in [process]+process.children(recursive=True):
                try:rss+=p.memory_info().rss
                except psutil.NoSuchProcess:pass
        except psutil.NoSuchProcess:pass
        peak=max(rss,peak)
        elapsed=time.monotonic()-start
        if elapsed>=6600 or rss>=24*(1024**3):
            raise RuntimeError('CPU job budget exceeded')
        if elapsed>=next_print:
            print('cpu_job_seconds',round(elapsed),'rss_GiB',round(rss/1024**3,2),flush=True)
            next_print=elapsed+60
        time.sleep(2)
    if child.returncode!=0:raise RuntimeError('CPU subprocess failed: '+str(child.returncode))
finally:
    if child.poll() is None:
        for p in process.children(recursive=True):
            try:p.kill()
            except psutil.NoSuchProcess:pass
        child.kill()
        child.wait()
    receipt={'job':'main','versions':actual,'returncode':child.returncode,'seconds':time.monotonic()-start,'peak_rss_bytes':peak,'config_sha256':digest(target/'experiment-config.json')}
    (out/'cloud-receipt.json').write_text(json.dumps(receipt,indent=2))
    with zipfile.ZipFile('/kaggle/working/exp053-results.zip','w',zipfile.ZIP_DEFLATED) as z:
        for p in out.rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(out).as_posix())
